### Rename files within a directory

In [ ]:
import os
import re
from pathlib import Path

In [ ]:
directory = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\852834\852834_2026-07-28_08-04-35\slap2\dynamic_data")  # <-- change this
dry_run = False  # set to False to actually rename files

In [ ]:
pattern = re.compile(r"^structure_20260728?")

In [ ]:
for path in directory.iterdir():
    if not path.is_file():
        continue

    old_name = path.name
    new_name = pattern.sub("acquisition", old_name)

    if new_name == old_name:
        continue  # nothing to change

    old_path = path
    new_path = path.with_name(new_name)

    if dry_run:
        print(f"[DRY RUN] {old_name}  ->  {new_name}")
    else:
        if new_path.exists():
            raise FileExistsError(f"Target already exists: {new_path}")
        old_path.rename(new_path)
        print(f"Renamed: {old_name} -> {new_name}")

### Interpret directory structure

In [ ]:
from pathlib import Path

paths = []

root = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f")

for p in root.rglob("*"):
    rel = p.relative_to(root)
    paths.append((str(rel), "dir" if p.is_dir() else "file"))

In [ ]:
paths

In [ ]:
# Save as a simple TSV
out = root / "glutamate_target_tree_manifest.tsv"
out.write_text("\n".join(f"{t}\t{kind}" for t, kind in sorted(paths)))
print("Wrote:", out)

### Create new directory structure

In [ ]:
from pathlib import Path
import csv

def load_directory_schema(example_manifest_path):
    """
    Reads the example manifest and returns a set of relative directory paths
    *excluding* the top-level session folder.
    """
    example_manifest_path = Path(example_manifest_path)

    dir_paths = set()

    with example_manifest_path.open(newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for rel_path, kind in reader:
            if kind != "dir":
                continue

            p = Path(rel_path)

            # Skip top-level session directory itself
            if len(p.parts) <= 1:
                continue

            # Remove the session-level folder
            dir_paths.add(Path(*p.parts[1:]))

    return sorted(dir_paths)


def create_directory_tree(target_root, dir_schema, dry_run=False):
    """
    Creates the directory tree under target_root using the schema.
    """
    target_root = Path(target_root)

    for rel_dir in dir_schema:
        full_path = target_root / rel_dir
        if dry_run:
            print(f"[DRY RUN] mkdir {full_path}")
        else:
            full_path.mkdir(parents=True, exist_ok=True)


def apply_schema_to_many_roots(
    example_manifest,
    target_roots,
    dry_run=False,
):
    """
    Applies the example directory schema to many dataset roots.
    """
    dir_schema = load_directory_schema(example_manifest)

    for root in target_roots:
        print(f"\n== Creating structure under: {root}")
        create_directory_tree(root, dir_schema, dry_run=dry_run)

In [ ]:
example_manifest = r"\\allen\aind\scratch\ophys\Andrew\example_tree_manifest.tsv"

target_roots = [
    r"\\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033"]

apply_schema_to_many_roots(
    example_manifest=example_manifest,
    target_roots=target_roots,
    dry_run=False,   # ← set False once you're happy
)

### Create new directory and automatically move data

In [ ]:
from __future__ import annotations

import csv
import re
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple


# =========================
# Options + logging
# =========================

@dataclass(frozen=True)
class Opt:
    dry_run: bool = True
    plan_csv: str = "reorg_move_plan.csv"
    collision_suffix_fmt: str = "_{i:03d}"

    # Timestamp selection
    raw_pick: str = "earliest"   # earliest/latest among acquisition/ref_stack tifs
    proc_pick: str = "latest"    # latest among ExperimentSummary files

    # SLAP2 discovery
    slap2_data_dirname: str = "SLAP2_data"
    slap2_run_regex: str = r"^slap2_.*"
    tif_exts: Tuple[str, ...] = (".tif", ".tiff")

    # If provided, we can infer the harp folder name from the example tree (e.g. VCO1_Behavior.harp)
    infer_harp_name_from_example: bool = True

    # Behavior camera naming
    video_name_to_camera: Tuple[Tuple[str, str], ...] = (
        ("body_camera.avi", "BodyCamera"),
        ("eye_camera.avi", "EyeCamera"),
        ("face_camera.avi", "FaceCamera"),
    )
    json_name_to_camera: Tuple[Tuple[str, str], ...] = (
        ("BodyCamera.json", "BodyCamera"),
        ("EyeCamera.json", "EyeCamera"),
        ("FaceCamera.json", "FaceCamera"),
    )


def log(msg: str) -> None:
    print(msg)


# =========================
# Safe FS helpers
# =========================

def ensure_unique_dest(dst: Path, fmt: str) -> Path:
    if not dst.exists():
        return dst
    i = 1
    while True:
        cand = dst.parent / f"{dst.name}{fmt.format(i=i)}"
        if not cand.exists():
            return cand
        i += 1


def safe_mkdir(p: Path, opt: Opt, plan: List[Dict[str, str]], note: str = "") -> None:
    plan.append({"action": "mkdir", "src": "", "dst": str(p), "note": note})
    if opt.dry_run:
        log(f"[DRY RUN] mkdir  {p}" + (f"  ({note})" if note else ""))
    else:
        p.mkdir(parents=True, exist_ok=True)


def safe_move(src: Path, dst: Path, opt: Opt, plan: List[Dict[str, str]], note: str = "") -> None:
    if not src.exists():
        plan.append({"action": "skip", "src": str(src), "dst": str(dst), "note": "missing src | " + note})
        log(f"[SKIP] missing src: {src} -> {dst} ({note})")
        return

    final_dst = ensure_unique_dest(dst, opt.collision_suffix_fmt)
    if final_dst != dst:
        note = (note + " | " if note else "") + f"collision-> {final_dst.name}"

    plan.append({"action": "move", "src": str(src), "dst": str(final_dst), "note": note})
    log(f"{'[DRY RUN] ' if opt.dry_run else ''}move   {src} -> {final_dst}" + (f"  ({note})" if note else ""))

    if opt.dry_run:
        return

    final_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(final_dst))


def safe_move_children(src_dir: Path, dst_dir: Path, opt: Opt, plan: List[Dict[str, str]],
                       note: str = "", exclude_names: Optional[set] = None) -> None:
    exclude_names = exclude_names or set()
    if not src_dir.exists() or not src_dir.is_dir():
        plan.append({"action": "skip", "src": str(src_dir), "dst": str(dst_dir), "note": "missing dir | " + note})
        log(f"[SKIP] missing dir: {src_dir} -> {dst_dir} ({note})")
        return

    safe_mkdir(dst_dir, opt, plan, note=f"ensure dst ({note})")
    for child in src_dir.iterdir():
        if child.name in exclude_names:
            log(f"[SKIP] excluded: {child} ({note})")
            continue
        safe_move(child, dst_dir / child.name, opt, plan, note=note)


# =========================
# Timestamp + discovery helpers
# =========================

def pick_mtime(paths: List[Path], pick: str) -> Optional[float]:
    if not paths:
        return None
    mt = [p.stat().st_mtime for p in paths]
    if pick == "earliest":
        return min(mt)
    if pick == "latest":
        return max(mt)
    raise ValueError(f"Unknown pick={pick!r}")


def fmt_ts(ts: float) -> str:
    return datetime.fromtimestamp(ts).strftime("%Y-%m-%d_%H-%M-%S")


def infer_mouse_id(root: Path) -> Optional[str]:
    m = re.search(r"(\d{5,7})", root.name)
    return m.group(1) if m else None


def find_slap2_data_dir(root: Path, opt: Opt) -> Optional[Path]:
    cand = root / "imaging_data" / opt.slap2_data_dirname
    if cand.exists():
        return cand
    target = opt.slap2_data_dirname.lower()
    for d in root.rglob("*"):
        if d.is_dir() and d.name.lower() == target:
            return d
    return None


def find_slap2_runs(slap2_data_dir: Path, opt: Opt) -> List[Path]:
    pat = re.compile(opt.slap2_run_regex, re.IGNORECASE)
    return [d for d in slap2_data_dir.iterdir() if d.is_dir() and pat.match(d.name)]


def choose_best_run(runs: List[Path]) -> Optional[Path]:
    if not runs:
        return None

    def score(run: Path) -> Tuple[int, int]:
        acq = list(run.rglob("acquisition_*.tif"))
        tifs = list(run.rglob("*.tif"))
        return (len(acq), len(tifs))

    return sorted(runs, key=score, reverse=True)[0]


def find_behavior_data_dir(root: Path) -> Optional[Path]:
    cand = root / "behavior_data"
    if cand.exists() and cand.is_dir():
        return cand
    for d in root.rglob("*"):
        if d.is_dir() and d.name.lower() == "behavior_data":
            return d
    return None


# =========================
# Example manifest helper: infer harp folder name
# =========================

def infer_harp_dst_name_from_example(example_manifest: Optional[Path]) -> str:
    """
    Find <raw_top>\\behavior\\<something>.harp in example manifest.
    Fallback: Behavior.harp
    """
    if example_manifest is None or not example_manifest.exists():
        return "Behavior.harp"

    rows: List[Tuple[str, str]] = []
    with example_manifest.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                rows.append((parts[0], parts[1]))

    tops = sorted({p.split("\\")[0] for p, _ in rows})
    raw_pat = re.compile(r"^\d+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")
    raw_top = next((t for t in tops if raw_pat.match(t)), "")
    if not raw_top:
        return "Behavior.harp"

    pat = re.compile(rf"^{re.escape(raw_top)}\\behavior\\([^\\]+\.harp)$", re.IGNORECASE)
    for p, typ in rows:
        if typ == "dir":
            m = pat.match(p)
            if m:
                return m.group(1)

    return "Behavior.harp"


# =========================
# Classification regexes (imaging)
# =========================

ACQ_RE = re.compile(r"^acquisition_.*\.(tif|tiff|dat|json|meta)$", re.IGNORECASE)
REF_RE = re.compile(r"^ref_stack_.*\.(tif|tiff|dat|meta)$", re.IGNORECASE)
MC_RE  = re.compile(r"^E\d+T\d+DMD\d+_.*", re.IGNORECASE)
TRIALTABLE_RE = re.compile(r"^trialTable\..*", re.IGNORECASE)


# =========================
# Cleanup: delete dirs that contain no files anywhere
# =========================

def dir_contains_any_file(d: Path) -> bool:
    if not d.exists() or not d.is_dir():
        return False
    for p in d.rglob("*"):
        if p.is_file():
            return True
    return False


def delete_dir_if_no_files(d: Path, opt: Opt, plan: List[Dict[str, str]], note: str = "") -> None:
    if not d.exists() or not d.is_dir():
        return
    if dir_contains_any_file(d):
        plan.append({"action": "keep", "src": str(d), "dst": "", "note": "contains files | " + note})
        log(f"[KEEP] {d} still contains files -> not deleting. ({note})")
        return

    plan.append({"action": "delete", "src": str(d), "dst": "", "note": note})
    if opt.dry_run:
        log(f"[DRY RUN] delete {d} ({note})")
        return
    shutil.rmtree(d)
    log(f"[OK] deleted {d} ({note})")


# =========================
# Behavior mover (with your requested fixes)
# =========================

def move_behavior_and_videos(
    dataset_root: Path,
    raw_session: Path,
    backup_session: Path,
    harp_dst_name: str,
    opt: Opt,
    plan: List[Dict[str, str]],
) -> None:
    """
    - Move .harp dir to raw_session/behavior/<harp_dst_name>
    - Pull extracted_files + bonsai_event_log.csv OUT of harp dir into raw_session/behavior/
    - Move behavior_data/behavior/* into raw_session/behavior/*
    - Move camera AVIs into raw_session/behavior-videos/<Cam>/
    - Move camera JSON metadata from SoftwareEvents into raw_session/behavior-videos/<Cam>/<Cam>.json
    - Any leftovers from behavior_data -> backup (then delete behavior_data if file-empty)
    """
    log("\n====================")
    log("BEHAVIOR MOVES")
    log("====================")

    behavior_data = find_behavior_data_dir(dataset_root)
    log(f"behavior_data detected: {behavior_data}")

    dst_behavior = raw_session / "behavior"
    dst_videos = raw_session / "behavior-videos"
    safe_mkdir(dst_behavior, opt, plan, "ensure behavior/")
    for _, cam in opt.video_name_to_camera:
        safe_mkdir(dst_videos / cam, opt, plan, f"ensure behavior-videos/{cam}")

    if behavior_data is None:
        log("[WARN] No behavior_data directory found. Skipping behavior moves.")
        return

    # 1) Move harp directory
    harp_src = behavior_data / "Behavior.harp"
    if not harp_src.exists():
        harp_dirs = [d for d in behavior_data.iterdir() if d.is_dir() and d.name.lower().endswith(".harp")]
        harp_src = harp_dirs[0] if harp_dirs else harp_src

    harp_dst = dst_behavior / harp_dst_name
    if harp_src.exists():
        safe_move(harp_src, harp_dst, opt, plan, note="harp dir -> behavior/<harp_dst_name>")
    else:
        log("[WARN] No *.harp directory found under behavior_data")

    # 2) Pull extracted_files + bonsai_event_log.csv OUT of harp folder into behavior/
    extracted_dst = dst_behavior / "extracted_files"
    bonsai_dst = dst_behavior / "bonsai_event_log.csv"

    if harp_dst.exists() and harp_dst.is_dir():
        ef = harp_dst / "extracted_files"
        if ef.exists() and ef.is_dir():
            safe_move(ef, extracted_dst, opt, plan, note="extracted_files -> behavior/ (sibling of harp)")
        else:
            nested = [d for d in harp_dst.rglob("extracted_files") if d.is_dir()]
            if nested:
                safe_move(nested[0], extracted_dst, opt, plan, note="nested extracted_files -> behavior/")

        bel = harp_dst / "bonsai_event_log.csv"
        if bel.exists() and bel.is_file():
            safe_move(bel, bonsai_dst, opt, plan, note="bonsai_event_log.csv -> behavior/ (sibling of harp)")
        else:
            candidates = [p for p in harp_dst.rglob("bonsai_event_log.csv") if p.is_file()]
            if candidates:
                safe_move(candidates[0], bonsai_dst, opt, plan, note="nested bonsai_event_log.csv -> behavior/")
    else:
        log("[INFO] harp destination folder not present yet (dry-run collisions may delay this).")

    # 3) Move behavior_data/behavior/* into raw_session/behavior/*
    beh_folder = behavior_data / "behavior"
    if beh_folder.exists() and beh_folder.is_dir():
        safe_move_children(beh_folder, dst_behavior, opt, plan, note="behavior_data/behavior/* -> behavior/*")
    else:
        log("[WARN] No behavior_data/behavior folder found")

    # 4) Route camera AVIs + camera JSON metadata (including from SoftwareEvents) into behavior-videos
    cam_video_map = {n.lower(): cam for n, cam in opt.video_name_to_camera}
    cam_json_map = {n.lower(): cam for n, cam in opt.json_name_to_camera}

    # 4a) AVIs anywhere under behavior_data
    for f in behavior_data.rglob("*"):
        if f.is_file() and f.name.lower() in cam_video_map:
            cam = cam_video_map[f.name.lower()]
            safe_move(f, dst_videos / cam / f.name, opt, plan, note="camera AVI -> behavior-videos/<Cam>/")

    # 4b) JSONs may be under raw_session/behavior/SoftwareEvents after move; also search behavior_data to be safe
    json_search_roots = [behavior_data, dst_behavior / "SoftwareEvents"]
    for sr in json_search_roots:
        if not sr.exists() or not sr.is_dir():
            continue
        for f in sr.rglob("*"):
            if f.is_file() and f.name.lower() in cam_json_map:
                cam = cam_json_map[f.name.lower()]
                safe_move(f, dst_videos / cam / f"{cam}.json", opt, plan, note="camera JSON -> behavior-videos/<Cam>/<Cam>.json")

    # 5) Any remaining stuff inside behavior_data goes to backup, then delete behavior_data if file-empty
    if behavior_data.exists():
        safe_move(behavior_data, backup_session / "behavior_data_remaining", opt, plan, note="leftover behavior_data -> backup")
        # If we moved it, the original path no longer exists; if it still exists (dry-run), leave it.


# =========================
# Imaging mover (works with your example-style mapping)
# =========================

def move_imaging_like_example(
    run_dir: Path,
    raw_session: Path,
    processed_session: Optional[Path],
    backup_session: Path,
    opt: Opt,
    plan: List[Dict[str, str]],
) -> None:
    """
    Routes:
      - acquisition_* + rigDescription.json -> raw/slap2/dynamic_data/
      - ref_stack_* + reference_stack dir -> raw/slap2/dynamic_data/reference_stack/
      - ExperimentSummary/* -> processed/source_extraction/ExperimentSummary/
      - ANNOTATIONS.mat -> processed/source_extraction/
      - E#T#DMD#_* -> processed/motion_correction/
      - trialTable.* -> processed root
      - everything else -> backup/unmapped_slap2/
    """
    log("\n====================")
    log("IMAGING MOVES")
    log("====================")

    raw_dyn = raw_session / "slap2" / "dynamic_data"
    raw_ref = raw_dyn / "reference_stack"

    safe_mkdir(raw_dyn, opt, plan, "ensure raw_dyn")
    safe_mkdir(raw_ref, opt, plan, "ensure raw_ref")

    if processed_session is not None:
        safe_mkdir(processed_session / "motion_correction", opt, plan, "ensure motion_correction")
        safe_mkdir(processed_session / "source_extraction" / "ExperimentSummary", opt, plan, "ensure ExperimentSummary dst")

    # Walk everything under run_dir
    all_items = sorted(run_dir.rglob("*"), key=lambda p: (p.is_dir(), str(p).lower()))

    for p in all_items:
        if p.is_dir():
            if p.name.lower() == "reference_stack":
                safe_move(p, raw_ref, opt, plan, note="reference_stack dir -> raw_ref")
            continue

        name = p.name
        parts_lower = [x.lower() for x in p.parts]

        if "experimentsummary" in parts_lower:
            if processed_session is not None:
                safe_move(p, processed_session / "source_extraction" / "ExperimentSummary" / name,
                          opt, plan, note="ES -> processed/source_extraction/ExperimentSummary")
            else:
                safe_move(p, backup_session / "ExperimentSummary" / name,
                          opt, plan, note="ES -> backup (no processed_session)")
            continue

        if TRIALTABLE_RE.match(name):
            if processed_session is not None:
                safe_move(p, processed_session / name, opt, plan, note="trialTable -> processed root")
            else:
                safe_move(p, backup_session / name, opt, plan, note="trialTable -> backup (no processed_session)")
            continue

        if MC_RE.match(name):
            if processed_session is not None:
                safe_move(p, processed_session / "motion_correction" / name, opt, plan, note="motion corr -> processed/motion_correction")
            else:
                safe_move(p, backup_session / "motion_correction" / name, opt, plan, note="motion corr -> backup (no processed_session)")
            continue

        if name.lower() == "annotations.mat":
            if processed_session is not None:
                safe_move(p, processed_session / "source_extraction" / name, opt, plan, note="ANNOTATIONS -> processed/source_extraction")
            else:
                safe_move(p, backup_session / name, opt, plan, note="ANNOTATIONS -> backup (no processed_session)")
            continue

        if ACQ_RE.match(name) or name.lower() == "rigdescription.json":
            safe_move(p, raw_dyn / name, opt, plan, note="acquisition/rigDescription -> raw_dyn")
            continue

        if REF_RE.match(name) or ("ref_stack" in name.lower()):
            safe_move(p, raw_ref / name, opt, plan, note="ref_stack -> raw_ref")
            continue

        safe_move(p, backup_session / "unmapped_slap2" / p.relative_to(run_dir), opt, plan, note="unmapped slap2 -> backup")

    # Move the now-mostly-empty run_dir container itself into backup for traceability
    safe_move(run_dir, backup_session / run_dir.name, opt, plan, note="run_dir container -> backup")


# =========================
# Top-level pipeline
# =========================

def restructure_data(
    dataset_root: str | Path,
    example_manifest_tsv: Optional[str | Path] = None,
    mouse_id: Optional[str] = None,
    opt: Opt = Opt(),
) -> Dict[str, object]:
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"dataset_root not found: {root}")

    example_manifest = Path(example_manifest_tsv) if example_manifest_tsv is not None else None

    if mouse_id is None:
        mouse_id = infer_mouse_id(root)
    if mouse_id is None:
        raise ValueError("Could not infer mouse_id from dataset_root folder name; pass mouse_id explicitly.")

    plan: List[Dict[str, str]] = []

    # --- Find SLAP2 run ---
    log("\n====================")
    log("DISCOVERY")
    log("====================")
    slap2_data = find_slap2_data_dir(root, opt)
    if slap2_data is None:
        raise FileNotFoundError(f"Could not find '{opt.slap2_data_dirname}' under {root}")

    runs = find_slap2_runs(slap2_data, opt)
    log(f"SLAP2_data dir: {slap2_data}")
    log(f"slap2_* runs found: {len(runs)}")
    for r in runs[:10]:
        log(f"  - {r}")

    run_dir = choose_best_run(runs)
    if run_dir is None:
        raise FileNotFoundError(f"No slap2_* run dirs found in {slap2_data}")
    log(f"Chosen run_dir: {run_dir}")

    # --- Derive timestamps ---
    raw_tifs = list(run_dir.rglob("acquisition_*.tif")) + list(run_dir.rglob("ref_stack_*.tif"))
    if not raw_tifs:
        raw_tifs = list(run_dir.rglob("*.tif"))
    raw_ts_f = pick_mtime(raw_tifs, opt.raw_pick)
    if raw_ts_f is None:
        raise FileNotFoundError(f"No tif files found under run_dir: {run_dir}")
    raw_ts = fmt_ts(raw_ts_f)

    es_dirs = [d for d in run_dir.rglob("ExperimentSummary") if d.is_dir()]
    es_files: List[Path] = []
    for d in es_dirs:
        es_files.extend([p for p in d.rglob("*") if p.is_file()])
    proc_ts = None
    if es_files:
        proc_ts_f = pick_mtime(es_files, opt.proc_pick)
        proc_ts = fmt_ts(proc_ts_f) if proc_ts_f is not None else None

    log(f"raw_ts: {raw_ts} (tifs considered: {len(raw_tifs)})")
    log(f"ExperimentSummary dirs: {len(es_dirs)} | files: {len(es_files)} | processed_ts: {proc_ts}")

    # --- Destination sessions (directly under dataset root) ---
    raw_name = f"{mouse_id}_{raw_ts}"
    proc_name = f"{mouse_id}_{raw_ts}_slap2_{proc_ts}" if proc_ts else None
    backup_name = f"slap2_{mouse_id}_{raw_ts}_remaining_data_backup"

    raw_session = ensure_unique_dest(root / raw_name, opt.collision_suffix_fmt)
    processed_session = ensure_unique_dest(root / proc_name, opt.collision_suffix_fmt) if proc_name else None
    backup_session = ensure_unique_dest(root / backup_name, opt.collision_suffix_fmt)

    log("\n====================")
    log("DESTINATIONS")
    log("====================")
    log(f"RAW session: {raw_session}")
    log(f"PROCESSED session: {processed_session if processed_session else '[not created: no proc_ts]'}")
    log(f"BACKUP: {backup_session}")

    # --- Create core folder structure ---
    safe_mkdir(raw_session / "behavior", opt, plan, "core")
    safe_mkdir(raw_session / "behavior-videos" / "BodyCamera", opt, plan, "core")
    safe_mkdir(raw_session / "behavior-videos" / "EyeCamera", opt, plan, "core")
    safe_mkdir(raw_session / "behavior-videos" / "FaceCamera", opt, plan, "core")
    safe_mkdir(raw_session / "slap2" / "dynamic_data" / "reference_stack", opt, plan, "core")

    if processed_session is not None:
        safe_mkdir(processed_session / "motion_correction", opt, plan, "core")
        safe_mkdir(processed_session / "source_extraction" / "ExperimentSummary", opt, plan, "core")
    safe_mkdir(backup_session, opt, plan, "core")

    # --- Imaging moves ---
    move_imaging_like_example(run_dir, raw_session, processed_session, backup_session, opt, plan)

    # --- Behavior moves (with your latest fixes) ---
    harp_dst_name = "Behavior.harp"
    if opt.infer_harp_name_from_example and example_manifest is not None:
        harp_dst_name = infer_harp_dst_name_from_example(example_manifest)
    log(f"\nHarp destination folder name: {harp_dst_name}")

    move_behavior_and_videos(root, raw_session, backup_session, harp_dst_name, opt, plan)

    # --- Sweep remaining top-level non-session stuff into backup ---
    log("\n====================")
    log("SWEEP REMAINING TOP-LEVEL ITEMS -> BACKUP")
    log("====================")

    protected = {raw_session.name, backup_session.name, opt.plan_csv}
    if processed_session is not None:
        protected.add(processed_session.name)

    for child in list(root.iterdir()):
        if child.name in protected:
            continue
        # don't sweep the plan if it exists already
        if child.is_file() and child.name == opt.plan_csv:
            continue
        # move everything else into backup root
        safe_move(child, backup_session / "remaining_root_items" / child.name, opt, plan, note="sweep leftover root item -> backup")

    # --- Cleanup: delete original containers if they contain no files anywhere (empty dirs OK) ---
    log("\n====================")
    log("CLEANUP (DELETE IF NO FILES)")
    log("====================")

    # After sweeping, these often don't exist anymore; if they do and are file-empty, delete.
    for name in ("behavior_data", "imaging_data", "metadata"):
        delete_dir_if_no_files(root / name, opt, plan, note=f"delete {name} if file-empty")

    # also delete any other top-level directories that are file-empty (excluding sessions + backup)
    keep_names = {raw_session.name, backup_session.name}
    if processed_session is not None:
        keep_names.add(processed_session.name)

    for child in list(root.iterdir()):
        if child.is_dir() and child.name not in keep_names:
            delete_dir_if_no_files(child, opt, plan, note="top-level file-empty cleanup")

    # --- Write plan CSV ---
    plan_path = root / opt.plan_csv
    if opt.dry_run:
        log(f"\n[DRY RUN] would write plan CSV -> {plan_path}")
    else:
        with plan_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["action", "src", "dst", "note"])
            w.writeheader()
            w.writerows(plan)
        log(f"\n[OK] wrote plan CSV -> {plan_path}")

    return {
        "dataset_root": str(root),
        "mouse_id": mouse_id,
        "raw_ts": raw_ts,
        "processed_ts": proc_ts,
        "raw_session": str(raw_session),
        "processed_session": str(processed_session) if processed_session else None,
        "backup_session": str(backup_session),
        "plan_csv": str(plan_path),
        "dry_run": opt.dry_run,
        "n_moves": sum(1 for r in plan if r["action"] == "move"),
        "n_skips": sum(1 for r in plan if r["action"] == "skip"),
        "n_deletes": sum(1 for r in plan if r["action"] == "delete"),
    }


In [ ]:
dataset_root = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033"
example_manifest = r"\\allen\aind\scratch\ophys\Andrew\example_tree_manifest.tsv"  # optional but recommended
summary = restructure_data(
    dataset_root=dataset_root,
    example_manifest_tsv=example_manifest,
    mouse_id="826033",
    opt=Opt(dry_run=True)
)
summary

### Iterate over directories

In [ ]:
import re
import traceback
from pathlib import Path

# ---- EDIT THESE ----
DATA_DIR = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
EXAMPLE_MANIFEST = Path(r"\\allen\aind\scratch\ophys\Andrew\example_tree_manifest.tsv")
DRY_RUN = True  # True first, then False
# --------------------

# sanity check
assert "restructure_data" in globals() and callable(restructure_data), \
    "restructure_data() not found. Run the cell that defines it first."

def looks_already_restructured(session_dir: Path, mouse_id: str) -> bool:
    """
    Skip if session folder already contains a raw-session folder like:
        mouseID_yyyy-mm-dd_hh-mm-ss/slap2/dynamic_data
    """
    raw_pat = re.compile(rf"^{re.escape(mouse_id)}_\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}}$", re.IGNORECASE)
    try:
        for child in session_dir.iterdir():
            if child.is_dir() and raw_pat.match(child.name):
                if (child / "slap2" / "dynamic_data").is_dir():
                    return True
    except Exception:
        # if we can't inspect, don't skip—let it fail loudly
        return False
    return False

def looks_like_session_folder(folder_name: str, mouse_id: str) -> bool:
    """
    Only process folders that look like sessions:
      - start with YYYY-MM-DD_
      - contain the mouse_id somewhere
    """
    return (mouse_id in folder_name) and bool(re.match(r"^\d{4}-\d{2}-\d{2}_", folder_name))

results, skipped, failures = [], [], []

for mouse_folder in sorted(DATA_DIR.iterdir()):
    if not mouse_folder.is_dir():
        continue

    mouse_id = mouse_folder.name
    if not mouse_id.isdigit():
        continue

    print(f"\n=== mouse {mouse_id} ===")

    session_dirs = [p for p in sorted(mouse_folder.iterdir()) if p.is_dir()]
    session_dirs = [p for p in session_dirs if looks_like_session_folder(p.name, mouse_id)]

    for session_path in session_dirs:
        try:
            if looks_already_restructured(session_path, mouse_id):
                print(f"SKIP (already restructured): {session_path}")
                skipped.append((mouse_id, str(session_path), "already restructured"))
                continue

            print(f"\n-- processing: {session_path}")

            # Call your function. If your signature differs, adjust these kwargs.
            summary = restructure_data(
                dataset_root=str(session_path),
                mouse_id=mouse_id,
                example_manifest_tsv=str(EXAMPLE_MANIFEST) if EXAMPLE_MANIFEST.exists() else None,
                opt=Opt(dry_run=DRY_RUN),
            )

            results.append(summary)
            print("OK")

        except Exception as e:
            failures.append((mouse_id, str(session_path), repr(e)))
            print(f"FAILED: {session_path}")
            print(f"  Error: {e!r}")
            traceback.print_exc()

print("\n====================")
print("DONE")
print("====================")
print(f"Successes: {len(results)}")
print(f"Skipped:   {len(skipped)}")
print(f"Failures:  {len(failures)}")

if failures:
    print("\nFirst 10 failures:")
    for row in failures[:10]:
        print(row)

In [ ]:
import glob

In [ ]:
glob.glob(os.path.join(DATA_DIR,'**','**DESC_.mat**'),recursive=True)

In [ ]:
from pathlib import Path

ROOT = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_NAME = "reorg_move_plan.csv"

# ---- safety toggle ----
DRY_RUN = False   # set to False to actually delete
# ----------------------

paths = list(ROOT.rglob(TARGET_NAME))
print(f"Found {len(paths)} file(s) named '{TARGET_NAME}' under:\n  {ROOT}\n")

for p in paths:
    try:
        if DRY_RUN:
            print(f"[DRY RUN] would delete: {p}")
        else:
            p.unlink()
            print(f"[DELETED] {p}")
    except Exception as e:
        print(f"[FAILED] {p}  ({e!r})")

print("\nDone.")

In [ ]:
from pathlib import Path
import re
import shutil

ROOT = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
PATTERN = "DESC_*.mat"

DRY_RUN = False  # set False to actually move

# --- helpers ---

SESSION_DIR_RE = re.compile(r"^\d{4}-\d{2}-\d{2}_.+")   # e.g. 2025-10-06_803121 or 2026-01-13_814591_BigStacks
RAW_SESSION_RE = re.compile(r"^\d+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")  # e.g. 803121_2025-10-30_11-13-32

def is_session_root(p: Path) -> bool:
    return p.is_dir() and bool(SESSION_DIR_RE.match(p.name)) and p.parent.name.isdigit()

def find_session_root(desc_path: Path) -> Path | None:
    for parent in desc_path.parents:
        if is_session_root(parent):
            return parent
    return None

def choose_raw_session_folder(session_root: Path) -> Path | None:
    """
    Choose a raw session folder inside session_root that already contains slap2/dynamic_data.
    If multiple exist, pick the first (sorted).
    """
    candidates = []
    for child in session_root.iterdir():
        if child.is_dir() and RAW_SESSION_RE.match(child.name):
            if (child / "slap2" / "dynamic_data").is_dir():
                candidates.append(child)
    return sorted(candidates)[0] if candidates else None

def unique_dest(dst: Path) -> Path:
    if not dst.exists():
        return dst
    stem, suf = dst.stem, dst.suffix
    i = 1
    while True:
        cand = dst.with_name(f"{stem}_{i:03d}{suf}")
        if not cand.exists():
            return cand
        i += 1

# --- main ---

desc_files = list(ROOT.rglob(PATTERN))
print(f"Found {len(desc_files)} files matching {PATTERN} under:\n  {ROOT}\n")

moved = 0
skipped = 0
failed = 0

for src in sorted(desc_files):
    try:
        session_root = find_session_root(src)
        if session_root is None:
            print(f"[SKIP] couldn't infer session root for: {src}")
            skipped += 1
            continue

        raw_session = choose_raw_session_folder(session_root)
        if raw_session is None:
            print(f"[SKIP] no raw session folder with slap2/dynamic_data found in: {session_root}")
            skipped += 1
            continue

        dst_dir = raw_session / "slap2" / "dynamic_data"
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst = unique_dest(dst_dir / src.name)

        if DRY_RUN:
            print(f"[DRY RUN] move {src} -> {dst}")
        else:
            shutil.move(str(src), str(dst))
            print(f"[MOVED] {src} -> {dst}")
        moved += 1

    except Exception as e:
        print(f"[FAILED] {src} ({e!r})")
        failed += 1

print("\nDone.")
print(f"Moved:   {moved}")
print(f"Skipped: {skipped}")
print(f"Failed:  {failed}")
